# Learning Technique Recommender
A pipeline that takes a raw course syllabus and recommends the most effective study techniques based on historical grade data.

### Pipeline overview
1. **Data Analysis** — load a CSV of student records and compute average grades per technique per course type  
2. **Syllabus Classifier** — use TF-IDF + cosine similarity to map a syllabus to a course type  
3. **Recommender** — rank learning techniques by effectiveness for a course type  
4. **Integrated System** — combine classifier + recommender into one end-to-end `analyze_syllabus()` call  
5. **Demo** — run the full pipeline on example syllabi

In [ ]:
import io
from typing import Dict, List, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")

## Part 1 — Data Analysis
Upload a CSV with columns `Course Type`, `Learning Technique`, and `Grade of Module (%)`.  
If no file is uploaded (or running outside Colab), the pre-analyzed default dataset is used.

In [ ]:
# Pre-analyzed grade data (fallback / default)
DEFAULT_TECHNIQUE_GRADES: Dict[str, Dict[str, float]] = {
    "Applied Calculation-Driven Learning": {
        "Worked Example Analysis": 93.45,
        "Simulation & Visualization": 93.06,
        "Deliberate Practice": 91.63,
        "Reverse Engineering": 90.80,
        "Iterative Writing & Editing": 63.45,
        "Case Study Analysis": 61.15,
    },
    "Deep Conceptual Learning": {
        "Conceptual Mapping": 93.25,
        "Spaced Repetition": 92.46,
        "Feynman Technique": 92.42,
        "Active Recall": 92.33,
        "Project-Based Learning": 62.93,
        "Open-Ended Exploration": 60.41,
    },
    "Case-Based & Strategic Learning": {
        "Case Study Analysis": 93.97,
        "Experiential Learning": 92.98,
        "Comparative Analysis": 92.60,
        "First-Principles Thinking": 92.50,
        "Simulation & Visualization": 63.42,
        "Spaced Repetition": 60.42,
    },
    "Language & Communication-Based Learning": {
        "Immersive Practice": 93.14,
        "Iterative Writing & Editing": 92.70,
        "Storytelling Frameworks": 92.27,
        "Active Recall & Shadowing": 92.03,
        "Worked Example Analysis": 63.76,
        "Deliberate Practice": 61.82,
    },
    "Hands-On, Project-Based Learning": {
        "Incremental Skill Building": 93.67,
        "Project-Based Learning": 92.23,
        "Learn-By-Building": 91.65,
        "Work-Along & Solving": 91.50,
        "Spaced Repetition": 64.94,
        "Conceptual Mapping": 64.78,
    },
}

REQUIRED_COLUMNS = ["Course Type", "Learning Technique", "Grade of Module (%)"]


def compute_technique_grades(data: pd.DataFrame) -> Dict[str, Dict[str, float]]:
    """Compute average grade per (course type, technique) pair from a DataFrame."""
    missing = [c for c in REQUIRED_COLUMNS if c not in data.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    return {
        course_type: (
            group.groupby("Learning Technique")["Grade of Module (%)"]
            .mean()
            .round(2)
            .to_dict()
        )
        for course_type, group in data.groupby("Course Type")
    }


def load_data() -> Dict[str, Dict[str, float]]:
    """Upload a CSV in Colab or fall back to the default dataset."""
    if not IN_COLAB:
        print("Running outside Colab — using default pre-analyzed data.")
        return DEFAULT_TECHNIQUE_GRADES

    print("Upload your CSV (columns: Course Type, Learning Technique, Grade of Module (%))")
    try:
        uploaded = files.upload()
        for filename, content in uploaded.items():
            data = pd.read_csv(io.BytesIO(content))
            print(f"Loaded '{filename}' — {data.shape[0]:,} rows")
            grades = compute_technique_grades(data)
            print(f"Computed grades for {len(grades)} course types.")
            return grades
    except Exception as exc:
        print(f"Upload failed ({exc}) — falling back to default data.")

    return DEFAULT_TECHNIQUE_GRADES


def visualize_technique_effectiveness(technique_grades: Dict[str, Dict[str, float]]) -> None:
    """One bar chart per course type showing technique effectiveness."""
    n = len(technique_grades)
    fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]

    for ax, (course_type, techniques) in zip(axes, technique_grades.items()):
        items = sorted(techniques.items(), key=lambda x: x[1], reverse=True)
        names, grades = zip(*items)
        colors = [
            "#2E7D32" if g > 90 else "#1976D2" if g > 80 else "#FFA000" if g > 70 else "#D32F2F"
            for g in grades
        ]
        ax.bar(names, grades, color=colors)
        ax.set_title(course_type, fontsize=12, fontweight="bold")
        ax.set_ylabel("Average Grade (%)")
        ax.set_ylim(50, 100)
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        for j, g in enumerate(grades):
            ax.text(j, g + 0.5, f"{g:.1f}%", ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()


# Load data and visualize
technique_grades = load_data()
visualize_technique_effectiveness(technique_grades)

## Part 2 — Syllabus Classifier
Uses TF-IDF vectorization and cosine similarity to score how closely a syllabus matches each course type description.  
Returns a ranked list of `(course_type, confidence_score)` pairs.

In [ ]:
class SyllabusClassifier:
    """
    Classifies raw syllabus text into one of the known course types
    using TF-IDF + cosine similarity against hand-crafted type descriptions.
    """

    COURSE_TYPE_DESCRIPTIONS = {
        "Applied Calculation-Driven Learning": (
            "mathematical formulas equations calculations quantitative numerical computational "
            "statistics models algorithms mathematics physics engineering economics finance "
            "problem sets computational exercises solving equations"
        ),
        "Deep Conceptual Learning": (
            "theoretical frameworks abstract concepts principles critical thinking theories "
            "conceptual analysis philosophy theoretical physics pure mathematics theoretical "
            "computer science theories concepts principles models theoretical essays"
        ),
        "Case-Based & Strategic Learning": (
            "case studies real-world scenarios strategic situations decision-making frameworks "
            "business strategy management law medicine policy cases strategic analysis "
            "frameworks decision scenarios strategic plans case analyses"
        ),
        "Language & Communication-Based Learning": (
            "language skills communication techniques writing speaking rhetoric expression "
            "literature languages journalism communication studies essays presentations "
            "writing exercises communication projects"
        ),
        "Hands-On, Project-Based Learning": (
            "hands-on projects building creating practical application learning by doing "
            "engineering design programming art music laboratory sciences projects "
            "practical exercises build systems project deliverables portfolios demonstrations"
        ),
    }

    def __init__(self):
        self._course_types = list(self.COURSE_TYPE_DESCRIPTIONS.keys())
        self._vectorizer = TfidfVectorizer(stop_words="english")
        corpus = list(self.COURSE_TYPE_DESCRIPTIONS.values())
        self._type_vectors = self._vectorizer.fit_transform(corpus)

    def classify(self, syllabus_text: str) -> List[Tuple[str, float]]:
        """
        Return (course_type, confidence) pairs sorted by confidence descending.
        Confidence is cosine similarity in [0, 1].
        """
        syllabus_vector = self._vectorizer.transform([syllabus_text])
        scores = {
            ct: float(cosine_similarity(self._type_vectors[i], syllabus_vector)[0][0])
            for i, ct in enumerate(self._course_types)
        }
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

## Part 3 — Learning Technique Recommender
Stores the per-course-type grade data and exposes methods to:
- **predict** the single best technique for a course type
- **rank** all techniques by effectiveness
- **visualize** technique effectiveness as a bar chart
- **retrain** from a new DataFrame

In [ ]:
class LearningTechniqueRecommender:
    """
    Recommends learning techniques for a course type based on historical grade data.
    Initialized with a technique_grades dict; can be retrained from a new DataFrame.
    """

    def __init__(self, technique_grades: Dict[str, Dict[str, float]]):
        self.technique_grades = technique_grades
        self._refresh()

    # ── public API ────────────────────────────────────────────────────────────

    def predict(self, course_type: str) -> str:
        """Return the single best technique for a course type."""
        self._check(course_type)
        return self._best[course_type]

    def predict_with_grade(self, course_type: str) -> Dict[str, Union[str, float]]:
        """Return the best technique and its expected average grade."""
        technique = self.predict(course_type)
        return {"technique": technique, "expected_grade": self.technique_grades[course_type][technique]}

    def get_all_techniques_ranked(self, course_type: str) -> List[Dict[str, Union[str, float]]]:
        """Return all techniques for a course type sorted by effectiveness (best first)."""
        self._check(course_type)
        return [
            {"technique": t, "expected_grade": g}
            for t, g in sorted(
                self.technique_grades[course_type].items(), key=lambda x: x[1], reverse=True
            )
        ]

    def train(self, data: pd.DataFrame) -> None:
        """Retrain from a DataFrame with Course Type, Learning Technique, Grade columns."""
        self.technique_grades = compute_technique_grades(data)
        self._refresh()

    def visualize_course_type(self, course_type: str, figsize: Tuple = (10, 5)) -> None:
        """Bar chart of all techniques for a single course type."""
        self._check(course_type)
        ranked = self.get_all_techniques_ranked(course_type)
        names = [r["technique"] for r in ranked]
        grades = [r["expected_grade"] for r in ranked]
        colors = [
            "#2E7D32" if g > 90 else "#1976D2" if g > 80 else "#FFA000" if g > 70 else "#D32F2F"
            for g in grades
        ]
        fig, ax = plt.subplots(figsize=figsize)
        ax.bar(names, grades, color=colors)
        ax.set_title(f"Technique Effectiveness — {course_type}", fontsize=13, fontweight="bold")
        ax.set_ylabel("Average Grade (%)")
        ax.set_ylim(50, 100)
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        for i, g in enumerate(grades):
            ax.text(i, g + 0.5, f"{g:.1f}%", ha="center", fontsize=9)
        plt.tight_layout()
        plt.show()

    # ── properties ────────────────────────────────────────────────────────────

    @property
    def available_course_types(self) -> List[str]:
        return list(self.technique_grades.keys())

    @property
    def all_learning_techniques(self) -> List[str]:
        return sorted({t for techs in self.technique_grades.values() for t in techs})

    # ── internals ─────────────────────────────────────────────────────────────

    def _refresh(self) -> None:
        self._best: Dict[str, str] = {
            ct: max(techs.items(), key=lambda x: x[1])[0]
            for ct, techs in self.technique_grades.items()
        }

    def _check(self, course_type: str) -> None:
        if course_type not in self.technique_grades:
            raise ValueError(
                f"Unknown course type '{course_type}'. "
                f"Available: {self.available_course_types}"
            )

## Part 4 — Integrated System
`LearningStyleSystem` wires the classifier and recommender together.  
Call `analyze_syllabus(text)` to get ranked technique recommendations from raw syllabus text.

**Combined score formula:** `0.5 × course_type_match + 0.5 × (technique_grade / 100)`

In [ ]:
class LearningStyleSystem:
    """
    End-to-end pipeline: raw syllabus text → ranked learning technique recommendations.
    """

    def __init__(self, technique_grades: Dict[str, Dict[str, float]]):
        self.classifier = SyllabusClassifier()
        self.recommender = LearningTechniqueRecommender(technique_grades)

    def analyze_syllabus(self, syllabus_text: str, top_n: int = 5) -> Dict:
        """
        Classify the syllabus, then rank techniques by a combined score:
            combined = 0.5 * course_match + 0.5 * (expected_grade / 100)
        Returns the top_n techniques across all course types.
        """
        course_type_scores = self.classifier.classify(syllabus_text)

        results = []
        for course_type, match_score in course_type_scores:
            for item in self.recommender.get_all_techniques_ranked(course_type):
                combined = match_score * 0.5 + (item["expected_grade"] / 100) * 0.5
                results.append({
                    "technique": item["technique"],
                    "expected_grade": item["expected_grade"],
                    "course_type": course_type,
                    "course_match": round(match_score * 100, 1),
                    "combined_score": round(combined * 100, 1),
                })

        results.sort(key=lambda x: x["combined_score"], reverse=True)
        return {"course_type_scores": course_type_scores, "top_techniques": results[:top_n]}

    def print_results(self, results: Dict) -> None:
        """Pretty-print analysis results."""
        print("\nCourse Type Classification:")
        for i, (ct, score) in enumerate(results["course_type_scores"], 1):
            print(f"  {i}. {ct}: {score * 100:.1f}%")

        medals = {1: "🥇", 2: "🥈", 3: "🥉"}
        print("\nTop Recommended Learning Techniques:")
        for i, t in enumerate(results["top_techniques"], 1):
            label = medals.get(i, f"{i}.")
            print(f"  {label} {t['technique']} — Combined Score: {t['combined_score']}%")
            print(f"     Course type : {t['course_type']} (match: {t['course_match']}%)")
            print(f"     Expected grade: {t['expected_grade']}%")

    def visualize_results(self, results: Dict) -> None:
        """Side-by-side bar charts: course type confidence + top techniques."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        labels = [ct for ct, _ in results["course_type_scores"]]
        scores = [round(s * 100, 1) for _, s in results["course_type_scores"]]
        ax1.bar(labels, scores, color="steelblue")
        ax1.set_title("Course Type Classification Confidence", fontweight="bold")
        ax1.set_ylabel("Confidence (%)")
        ax1.tick_params(axis="x", rotation=45)
        ax1.grid(axis="y", linestyle="--", alpha=0.7)
        for i, v in enumerate(scores):
            ax1.text(i, v + 0.5, f"{v}%", ha="center", fontsize=9)

        techs = [t["technique"] for t in results["top_techniques"]]
        t_scores = [t["combined_score"] for t in results["top_techniques"]]
        ax2.bar(techs, t_scores, color="mediumseagreen")
        ax2.set_title("Top Recommended Learning Techniques", fontweight="bold")
        ax2.set_ylabel("Combined Score (%)")
        ax2.tick_params(axis="x", rotation=45)
        ax2.grid(axis="y", linestyle="--", alpha=0.7)
        for i, v in enumerate(t_scores):
            ax2.text(i, v + 0.5, f"{v}%", ha="center", fontsize=9)

        plt.tight_layout()
        plt.show()

## Part 5 — Demo
Run the full pipeline on four example syllabi spanning all course types.  
Replace any syllabus text below with your own to get a custom recommendation.

In [ ]:
EXAMPLE_SYLLABI = {
    "Calculus III": """
        MATH 301: Calculus III
        Multivariate calculus: partial derivatives, multiple integrals, vector calculus.
        Students solve complex mathematical problems using equations, formulas, and
        computational techniques.
        Assessment: Problem sets (40%), Two midterms (30%), Final exam (30%)
    """,
    "Introduction to Philosophy": """
        PHIL 201: Introduction to Philosophy
        Major philosophical theories and frameworks explored through critical reading.
        Topics: epistemology, metaphysics, ethics, consciousness, abstract reasoning.
        Assessment: Reading responses (30%), Participation (20%), Essays (30%), Paper (20%)
    """,
    "Strategic Management": """
        BUS 405: Strategic Management
        Case study analysis, competitive strategy, and organizational decision-making.
        Students develop strategic plans and business recommendations from real-world scenarios.
        Assessment: Case analyses (40%), Strategic plan (30%), Final exam (20%), Participation (10%)
    """,
    "Full-Stack Web Development": """
        CS 310: Full-Stack Web Development
        Hands-on project-based course: HTML, CSS, JavaScript, and backend frameworks.
        Students build and ship real applications through incremental project deliverables.
        Assessment: Projects (60%), Labs (20%), Participation (20%)
    """,
}

system = LearningStyleSystem(technique_grades)

for title, syllabus in EXAMPLE_SYLLABI.items():
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")
    results = system.analyze_syllabus(syllabus)
    system.print_results(results)
    system.visualize_results(results)